# Mutual Fund Analytics — Exploratory Data Analysis

## Project Title
**Indian Mutual Fund Industry Analytics (2022–2025)**

## Objective
Perform end-to-end exploratory data analysis on cleaned AMFI-style mutual fund datasets to uncover industry trends, investor behaviour patterns, fund performance dynamics, and portfolio allocation insights through statistical profiling and professional visualizations.

## Dataset Description

| # | Dataset | Records | Description |
|---|---------|---------|-------------|
| 01 | Fund Master | 40 schemes | Scheme metadata — fund house, category, plan, expense ratio |
| 02 | NAV History | ~46K rows | Daily Net Asset Value time series per scheme |
| 03 | AUM by Fund House | 90 rows | Quarterly AUM snapshots across 10 AMCs |
| 04 | Monthly SIP Inflows | 48 months | Industry SIP inflows, accounts, and YoY growth |
| 05 | Category Inflows | 144 rows | Monthly net inflows by fund category |
| 06 | Industry Folio Count | 21 snapshots | Total and category-wise investor folios |
| 07 | Scheme Performance | 40 schemes | Returns, risk metrics, AUM, and ratings |
| 08 | Investor Transactions | ~33K rows | SIP/Lumpsum/Redemption records with demographics |
| 09 | Portfolio Holdings | 322 rows | Stock-level sector weights across schemes |
| 10 | Benchmark Indices | ~8K rows | Daily closing values for 7 benchmark indices |

All datasets are sourced from cleaned CSV files in `../data/processed/`.

## Business Questions

1. How have mutual fund NAVs trended during the 2023 bull run and 2024 market correction?
2. Which fund houses dominate industry AUM, and how has market share evolved year-over-year?
3. Is retail SIP participation accelerating, and what is the trajectory of monthly inflows?
4. Which fund categories attract the most net inflows across months?
5. What is the demographic profile of mutual fund investors (age, gender, geography)?
6. How concentrated is SIP investment across T30 vs B30 cities?
7. How fast is the industry folio base growing?
8. How correlated are daily returns across major schemes?
9. What sectors dominate mutual fund portfolios?
10. Which schemes deliver the best risk-adjusted returns, and how do they compare to benchmarks?


In [ ]:
# =============================================================================
# 1. Import Libraries & Configure Plotting Style
# =============================================================================
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import seaborn as sns
from IPython.display import display

warnings.filterwarnings("ignore")

# Paths relative to notebooks/ directory
NOTEBOOK_DIR = Path(".").resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent
DATA_DIR = PROJECT_ROOT / "data" / "processed"
CHARTS_DIR = PROJECT_ROOT / "reports" / "charts"
CHARTS_DIR.mkdir(parents=True, exist_ok=True)

# Matplotlib / Seaborn styling
plt.style.use("seaborn-v0_8-whitegrid")
sns.set_theme(style="whitegrid", context="talk", palette="deep")
plt.rcParams.update({
    "figure.figsize": (12, 6),
    "figure.dpi": 120,
    "axes.titlesize": 14,
    "axes.titleweight": "bold",
    "axes.labelsize": 12,
    "legend.fontsize": 10,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "font.family": "sans-serif",
})

# Plotly template
PLOTLY_TEMPLATE = "plotly_white"
PLOTLY_COLORS = px.colors.qualitative.Set2

print(f"Data directory : {DATA_DIR}")
print(f"Charts directory: {CHARTS_DIR}")


In [ ]:
# =============================================================================
# Reusable Helper Functions
# =============================================================================

def save_matplotlib_figure(fig, filename: str) -> Path:
    """Save a Matplotlib/Seaborn figure as PNG."""
    filepath = CHARTS_DIR / filename
    fig.savefig(filepath, dpi=150, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print(f"Saved PNG  -> {filepath.name}")
    return filepath


def save_plotly_figure(fig, filename: str) -> Path:
    """Save a Plotly figure as interactive HTML."""
    filepath = CHARTS_DIR / filename
    fig.write_html(str(filepath), include_plotlyjs="cdn")
    print(f"Saved HTML -> {filepath.name}")
    return filepath


def explore_dataset(df: pd.DataFrame, name: str) -> None:
    """Print standard EDA summary for a single dataset."""
    print("=" * 70)
    print(f"DATASET: {name}")
    print("=" * 70)
    print(f"\nShape: {df.shape[0]:,} rows x {df.shape[1]} columns")
    print(f"\nColumns:\n{list(df.columns)}")
    print(f"\nData Types:\n{df.dtypes}")
    missing = df.isnull().sum()
    missing_pct = (missing / len(df) * 100).round(2)
    missing_table = pd.DataFrame({"missing_count": missing, "missing_pct": missing_pct})
    missing_rows = missing_table[missing_table["missing_count"] > 0]
    print(f"\nMissing Values:\n{missing_rows if not missing_rows.empty else 'None'}")
    print(f"\nDuplicate Rows: {df.duplicated().sum():,}")
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    if len(numeric_cols) > 0:
        print(f"\nSummary Statistics (numeric):\n{df[numeric_cols].describe().round(2).T}")
    print(f"\nFirst 5 Rows:\n{df.head()}")
    print()


def data_quality_report(datasets: dict) -> pd.DataFrame:
    """Build a consolidated missing-value and duplicate summary table."""
    rows = []
    for name, df in datasets.items():
        rows.append({
            "dataset": name,
            "rows": len(df),
            "columns": df.shape[1],
            "missing_cells": int(df.isnull().sum().sum()),
            "missing_pct": round(df.isnull().sum().sum() / df.size * 100, 2),
            "duplicate_rows": int(df.duplicated().sum()),
        })
    return pd.DataFrame(rows).sort_values("dataset")


In [ ]:
# =============================================================================
# 2. Load All Cleaned Datasets
# =============================================================================

DATA_FILES = {
    "fund_master": "01_fund_master_clean.csv",
    "nav_history": "02_nav_history_clean.csv",
    "aum_fund_house": "03_aum_by_fund_house_clean.csv",
    "monthly_sip": "04_monthly_sip_inflows_clean.csv",
    "category_inflows": "05_category_inflows_clean.csv",
    "industry_folio": "06_industry_folio_count_clean.csv",
    "scheme_performance": "07_scheme_performance_clean.csv",
    "investor_transactions": "08_investor_transactions_clean.csv",
    "portfolio_holdings": "09_portfolio_holdings_clean.csv",
    "benchmark_indices": "10_benching_indices_clean.csv",
}

datasets = {}
for key, filename in DATA_FILES.items():
    filepath = DATA_DIR / filename
    datasets[key] = pd.read_csv(filepath)
    print(f"Loaded {filename:<45} -> {datasets[key].shape[0]:>6,} rows")

# Convenience aliases
fund_master = datasets["fund_master"]
nav_history = datasets["nav_history"]
aum_fund_house = datasets["aum_fund_house"]
monthly_sip = datasets["monthly_sip"]
category_inflows = datasets["category_inflows"]
industry_folio = datasets["industry_folio"]
scheme_performance = datasets["scheme_performance"]
investor_transactions = datasets["investor_transactions"]
portfolio_holdings = datasets["portfolio_holdings"]
benchmark_indices = datasets["benchmark_indices"]

# Parse date columns
nav_history["date"] = pd.to_datetime(nav_history["date"])
aum_fund_house["date"] = pd.to_datetime(aum_fund_house["date"])
monthly_sip["month"] = pd.to_datetime(monthly_sip["month"])
category_inflows["month"] = pd.to_datetime(category_inflows["month"])
industry_folio["month"] = pd.to_datetime(industry_folio["month"])
investor_transactions["transaction_date"] = pd.to_datetime(investor_transactions["transaction_date"])
portfolio_holdings["portfolio_date"] = pd.to_datetime(portfolio_holdings["portfolio_date"])
benchmark_indices["date"] = pd.to_datetime(benchmark_indices["date"])
fund_master["launch_date"] = pd.to_datetime(fund_master["launch_date"], errors="coerce")

print(f"\nTotal datasets loaded: {len(datasets)}")


---

## 3. Initial Data Exploration

For each cleaned dataset we inspect shape, columns, data types, missing values, duplicates, summary statistics, and sample rows.


In [ ]:
# Run standard exploration for every dataset
for name, df in datasets.items():
    explore_dataset(df, name)


---

## 4. Data Quality Checks

Consolidated view of data completeness, duplicates, unique categorical values, and numerical summaries across all datasets.


In [ ]:
# Missing values & duplicate summary
quality_df = data_quality_report(datasets)
display(quality_df.style.background_gradient(subset=["missing_pct"], cmap="YlOrRd")
        .format({"rows": "{:,}", "missing_pct": "{:.2f}%"}))

# Duplicate rows detail
print("\n--- Duplicate Row Counts ---")
for name, df in datasets.items():
    dup = df.duplicated().sum()
    status = f"{dup:,} duplicates" if dup > 0 else "no duplicates"
    print(f"  {name}: {status}")

# Unique categories in key categorical columns
print("\n--- Unique Categories ---")
category_checks = {
    "fund_master": ["fund_house", "category", "plan", "risk_category"],
    "scheme_performance": ["fund_house", "category", "risk_grade"],
    "category_inflows": ["category"],
    "investor_transactions": ["transaction_type", "age_group", "gender", "city_tier", "state"],
    "portfolio_holdings": ["sector"],
    "benchmark_indices": ["index_name"],
}
for ds_name, cols in category_checks.items():
    df = datasets[ds_name]
    print(f"\n  [{ds_name}]")
    for col in cols:
        uniq = sorted(df[col].dropna().unique())
        preview = uniq[:8]
        suffix = "..." if len(uniq) > 8 else ""
        print(f"    {col}: {df[col].nunique()} unique -> {preview}{suffix}")

# Numerical summaries across all datasets
print("\n--- Consolidated Numerical Summaries ---")
for name, df in datasets.items():
    numeric = df.select_dtypes(include=[np.number])
    if numeric.empty:
        continue
    print(f"\n  {name}:")
    print(numeric.describe().round(2).T[["mean", "std", "min", "50%", "max"]].to_string())


---

## 5. Visualizations

Fifteen professional charts covering NAV trends, AUM growth, SIP flows, investor demographics, portfolio allocation, performance analytics, and benchmark comparison. All figures are exported to `../reports/charts/`.


### Chart 1 — Daily NAV Trend (All Schemes)

Interactive Plotly line chart of daily NAV merged with scheme names, highlighting the 2023 bull run and 2024 market correction.


In [ ]:
# Chart 1: Daily NAV Trend (Plotly)
nav_with_names = nav_history.merge(
    fund_master[["amfi_code", "scheme_name", "fund_house"]],
    on="amfi_code",
    how="left",
)

fig_nav = px.line(
    nav_with_names,
    x="date",
    y="nav",
    color="scheme_name",
    hover_data=["fund_house", "nav"],
    title="Daily NAV Trend — All Mutual Fund Schemes (2022–2025)",
    labels={"date": "Date", "nav": "Net Asset Value (INR)", "scheme_name": "Scheme"},
    template=PLOTLY_TEMPLATE,
    color_discrete_sequence=PLOTLY_COLORS,
)

fig_nav.add_vrect(
    x0="2023-01-01", x1="2023-12-31",
    fillcolor="rgba(46, 204, 113, 0.12)", line_width=0,
    annotation_text="2023 Bull Run", annotation_position="top left",
    annotation=dict(font_size=11, font_color="#27ae60"),
)
fig_nav.add_vrect(
    x0="2024-06-01", x1="2024-12-31",
    fillcolor="rgba(231, 76, 60, 0.12)", line_width=0,
    annotation_text="2024 Market Correction", annotation_position="top left",
    annotation=dict(font_size=11, font_color="#c0392b"),
)

fig_nav.update_layout(
    legend=dict(title="Scheme", orientation="v", yanchor="top", y=1, xanchor="left", x=1.02, font_size=9),
    hovermode="x unified",
    height=600,
    margin=dict(r=200),
)
fig_nav.update_xaxes(rangeslider_visible=True)
fig_nav.show()
save_plotly_figure(fig_nav, "01_daily_nav_trend.html")


#### Insight

- **Business insight:** NAV trajectories diverge sharply across categories — small-cap and mid-cap schemes show higher volatility but stronger cumulative gains during the 2023 bull run, while large-cap funds offer smoother paths attractive to conservative allocators.
- **Investment insight:** Investors who entered during the 2024 correction benefited from lower entry NAVs in quality large-cap funds, reinforcing the value of systematic investing through volatile periods.
- **Trend observation:** The broad upward slope from 2022–2023 followed by a mid-2024 pullback mirrors Nifty 50 behaviour, confirming high beta exposure in equity-oriented schemes.


### Chart 2 — AUM Growth by Fund House (Grouped Bar Chart)


In [ ]:
# Chart 2: AUM Growth — Fund House vs Year
aum_yearly = (
    aum_fund_house.assign(year=aum_fund_house["date"].dt.year)
    .groupby(["fund_house", "year"], as_index=False)["aum_lakh_crore"]
    .max()
    .sort_values(["year", "aum_lakh_crore"], ascending=[True, False])
)

fig_aum, ax_aum = plt.subplots(figsize=(14, 7))
sns.barplot(
    data=aum_yearly,
    x="year",
    y="aum_lakh_crore",
    hue="fund_house",
    ax=ax_aum,
    palette="tab10",
    edgecolor="white",
    linewidth=0.5,
)

sbi_peak = aum_yearly[(aum_yearly["fund_house"] == "SBI Mutual Fund") & (aum_yearly["year"] == 2025)]
if not sbi_peak.empty:
    peak_val = sbi_peak["aum_lakh_crore"].values[0]
    ax_aum.annotate(
        f"SBI MF: {peak_val:.2f} Lakh Cr\n(Industry Leader)",
        xy=(2025, peak_val),
        xytext=(2024.3, peak_val + 1.2),
        fontsize=10, fontweight="bold", color="#c0392b",
        arrowprops=dict(arrowstyle="->", color="#c0392b", lw=1.5),
        bbox=dict(boxstyle="round,pad=0.4", facecolor="#fdebd0", edgecolor="#c0392b"),
    )

ax_aum.set_title("AUM Growth by Fund House (Year-over-Year)", pad=15)
ax_aum.set_xlabel("Year")
ax_aum.set_ylabel("AUM (Lakh Crore INR)")
ax_aum.legend(title="Fund House", bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
plt.tight_layout()
save_matplotlib_figure(fig_aum, "02_aum_growth_by_fund_house.png")
plt.show()


#### Insight

- **Business insight:** SBI Mutual Fund's AUM nearly doubled from 6.3L Cr (2022) to 12.5L Cr (2025), cementing its position as India's largest AMC and widening the gap with HDFC and ICICI Prudential.
- **Investment insight:** Larger fund houses benefit from economies of scale — lower expense ratios and deeper research teams — making them default choices for first-time SIP investors.
- **Trend observation:** All top-10 AMCs show consistent AUM expansion, indicating industry-wide asset growth rather than share shifting between winners and losers.


### Chart 3 — Monthly SIP Inflows (Jan 2022 – Dec 2025)


In [ ]:
# Chart 3: Monthly SIP Inflows (Plotly)
sip_plot = monthly_sip.sort_values("month")

fig_sip = go.Figure()
fig_sip.add_trace(go.Scatter(
    x=sip_plot["month"],
    y=sip_plot["sip_inflow_crore"],
    mode="lines+markers",
    name="Monthly SIP Inflow",
    line=dict(color="#2980b9", width=2.5),
    marker=dict(size=5),
    hovertemplate="%{x|%b %Y}<br>%{y:,.0f} Cr<extra></extra>",
))

dec_2025 = sip_plot[sip_plot["month"] == "2025-12-01"]
if not dec_2025.empty:
    peak = dec_2025["sip_inflow_crore"].values[0]
    fig_sip.add_annotation(
        x=dec_2025["month"].values[0],
        y=peak,
        text=f"31,002 Cr<br>Dec 2025",
        showarrow=True,
        arrowhead=2,
        ax=0, ay=-50,
        font=dict(size=12, color="#c0392b", family="Arial Black"),
        bgcolor="rgba(255,255,255,0.85)",
        bordercolor="#c0392b",
    )

fig_sip.update_layout(
    title="Monthly SIP Inflows — Indian Mutual Fund Industry (Jan 2022 – Dec 2025)",
    xaxis_title="Month",
    yaxis_title="SIP Inflow (Crore INR)",
    template=PLOTLY_TEMPLATE,
    height=500,
    hovermode="x unified",
)
fig_sip.update_yaxes(tickformat=",")
fig_sip.show()
save_plotly_figure(fig_sip, "03_monthly_sip_inflows.html")


#### Insight

- **Business insight:** Monthly SIP inflows reached a record 31,002 crore in December 2025, signalling that retail systematic investing has become the primary growth engine for Indian mutual funds.
- **Investment insight:** Consistent SIP inflows create a structural bid for equities, partially cushioning market downturns and supporting long-term compounding for disciplined investors.
- **Trend observation:** The near-linear growth in SIP inflows from 2022–2025 reflects deepening financialisation of Indian household savings.


### Chart 4 — Category Inflow Heatmap


In [ ]:
# Chart 4: Category Inflow Heatmap
cat_pivot = category_inflows.pivot_table(
    index="category",
    columns=category_inflows["month"].dt.strftime("%b %Y"),
    values="net_inflow_crore",
    aggfunc="sum",
)

fig_heat, ax_heat = plt.subplots(figsize=(16, 8))
sns.heatmap(
    cat_pivot,
    annot=True,
    fmt=",d",
    cmap="RdYlGn",
    center=0,
    linewidths=0.5,
    ax=ax_heat,
    cbar_kws={"label": "Net Inflow (Crore INR)"},
    annot_kws={"size": 8},
)
ax_heat.set_title("Category-wise Net Inflows — Monthly Heatmap", pad=15)
ax_heat.set_xlabel("Month")
ax_heat.set_ylabel("Fund Category")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
save_matplotlib_figure(fig_heat, "04_category_inflow_heatmap.png")
plt.show()


#### Insight

- **Business insight:** Flexi Cap and Large Cap categories consistently attract the highest net inflows, while Sectoral/Thematic funds show lumpy, sentiment-driven flows.
- **Investment insight:** Categories with steady inflows (Flexi Cap, ELSS) tend to have more stable AUM bases, reducing redemption pressure during market stress.
- **Trend observation:** Liquid and Short Duration funds show periodic spikes, likely reflecting corporate treasury parking and rate-cycle positioning.


### Chart 5 — Investor Age Distribution


In [ ]:
# Chart 5: Investor Age Distribution (Pie Chart)
age_order = ["18-25", "26-35", "36-45", "46-55", "56+"]
age_counts = (
    investor_transactions["age_group"]
    .value_counts()
    .reindex(age_order)
    .dropna()
)

fig_age, ax_age = plt.subplots(figsize=(8, 8))
colors = sns.color_palette("Set2", len(age_counts))
_, _, autotexts = ax_age.pie(
    age_counts,
    labels=age_counts.index,
    autopct="%1.1f%%",
    startangle=90,
    colors=colors,
    pctdistance=0.75,
    wedgeprops=dict(width=0.55, edgecolor="white", linewidth=2),
)
ax_age.set_title("Investor Age Group Distribution", pad=15)
for t in autotexts:
    t.set_fontsize(10)
    t.set_fontweight("bold")
plt.tight_layout()
save_matplotlib_figure(fig_age, "05_investor_age_distribution.png")
plt.show()


#### Insight

- **Business insight:** The 26–35 age cohort dominates transaction volume (~41%), making young professionals the primary target segment for AMC digital marketing and app-based onboarding.
- **Investment insight:** Younger investors have longer investment horizons, favouring equity SIPs — a strategic fit for wealth-creation oriented schemes.
- **Trend observation:** The 56+ segment remains under-represented, suggesting an untapped opportunity for retirement-focused hybrid and debt funds.


### Chart 6 — SIP Amount Distribution by Age Group (Box Plot)


In [ ]:
# Chart 6: SIP Amount Box Plot by Age Group
sip_txns = investor_transactions[investor_transactions["transaction_type"] == "SIP"].copy()
age_order = ["18-25", "26-35", "36-45", "46-55", "56+"]

fig_box, ax_box = plt.subplots(figsize=(12, 6))
sns.boxplot(
    data=sip_txns,
    x="age_group",
    y="amount_inr",
    order=age_order,
    hue="age_group",
    palette="Blues",
    ax=ax_box,
    showfliers=False,
    linewidth=1.2,
    legend=False,
)
ax_box.set_title("SIP Transaction Amount Distribution by Age Group", pad=15)
ax_box.set_xlabel("Age Group")
ax_box.set_ylabel("SIP Amount (INR)")
ax_box.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
plt.tight_layout()
save_matplotlib_figure(fig_box, "06_sip_amount_by_age_group.png")
plt.show()


#### Insight

- **Business insight:** Median SIP amounts rise with age, peaking in the 36–45 bracket — AMCs should offer tiered SIP step-up plans aligned with career progression.
- **Investment insight:** The relatively narrow IQR across age groups suggests SIP is democratised at 5,000–10,000 ticket sizes, making it accessible to mass-market investors.
- **Trend observation:** Higher age groups show wider dispersion, indicating HNI investors using SIP as a disciplined allocation tool alongside lumpsum investments.


### Chart 7 — Gender Distribution (Count Plot)


In [ ]:
# Chart 7: Gender Distribution
fig_gender, ax_gender = plt.subplots(figsize=(8, 6))
gender_counts = investor_transactions["gender"].value_counts()
sns.countplot(
    data=investor_transactions,
    x="gender",
    order=gender_counts.index,
    hue="gender",
    palette={"Male": "#3498db", "Female": "#e74c3c"},
    ax=ax_gender,
    edgecolor="white",
    linewidth=1.5,
    legend=False,
)
for i, (label, count) in enumerate(gender_counts.items()):
    ax_gender.text(i, count + 200, f"{count:,}", ha="center", fontweight="bold", fontsize=11)
ax_gender.set_title("Investor Gender Distribution", pad=15)
ax_gender.set_xlabel("Gender")
ax_gender.set_ylabel("Number of Transactions")
plt.tight_layout()
save_matplotlib_figure(fig_gender, "07_gender_distribution.png")
plt.show()


#### Insight

- **Business insight:** Male investors account for roughly 2x the transaction volume of female investors, highlighting a gender gap that AMCs can address through targeted financial literacy campaigns.
- **Investment insight:** Closing the gender participation gap could unlock a significant new SIP subscriber base, particularly in Tier-2 and Tier-3 cities.
- **Trend observation:** The 2:1 ratio is consistent with broader Indian financial market participation patterns, though it is narrowing among younger cohorts.


### Chart 8 — State-wise SIP Investment (Horizontal Bar Chart)


In [ ]:
# Chart 8: State-wise SIP Investment
state_sip = (
    investor_transactions[investor_transactions["transaction_type"] == "SIP"]
    .groupby("state")["amount_inr"]
    .sum()
    .sort_values(ascending=True)
    .tail(15)
    .div(1e7)
)

fig_state, ax_state = plt.subplots(figsize=(12, 8))
bars = ax_state.barh(state_sip.index, state_sip.values, color=sns.color_palette("viridis", len(state_sip)))
ax_state.set_title("Top 15 States by SIP Investment Volume", pad=15)
ax_state.set_xlabel("Total SIP Investment (Crore INR)")
ax_state.set_ylabel("State")
for bar, val in zip(bars, state_sip.values):
    ax_state.text(
        bar.get_width() + 0.5,
        bar.get_y() + bar.get_height() / 2,
        f"{val:,.1f} Cr",
        va="center",
        fontsize=9,
    )
plt.tight_layout()
save_matplotlib_figure(fig_state, "08_statewise_sip_investment.png")
plt.show()


#### Insight

- **Business insight:** Punjab, Tamil Nadu, and Madhya Pradesh lead SIP volumes among the top states, indicating strong retail participation beyond the traditional metros of Maharashtra and Karnataka.
- **Investment insight:** Geographic diversification of SIP flows reduces concentration risk for AMCs and reflects growing financial awareness in non-metro India.
- **Trend observation:** North and South Indian states dominate the top tier, suggesting regional distributor networks and state-level financial inclusion initiatives are paying off.


### Chart 9 — T30 vs B30 City Distribution (Pie Chart)


In [ ]:
# Chart 9: T30 vs B30 Pie Chart
tier_counts = investor_transactions["city_tier"].value_counts()
tier_labels = {"T30": "Top 30 Cities", "B30": "Beyond Top 30"}

fig_tier, ax_tier = plt.subplots(figsize=(8, 8))
colors_tier = ["#2ecc71", "#f39c12"]
_, _, autotexts = ax_tier.pie(
    tier_counts.values,
    labels=[tier_labels.get(k, k) for k in tier_counts.index],
    autopct=lambda pct: f"{pct:.1f}%\n({int(pct / 100 * tier_counts.sum()):,})",
    startangle=140,
    colors=colors_tier,
    explode=(0.03, 0.03),
    wedgeprops=dict(edgecolor="white", linewidth=2),
)
ax_tier.set_title("Investor Distribution: T30 vs B30 Cities", pad=15)
plt.tight_layout()
save_matplotlib_figure(fig_tier, "09_t30_vs_b30_distribution.png")
plt.show()


#### Insight

- **Business insight:** T30 cities account for ~66% of transactions, but the B30 share (~34%) is growing rapidly — a key battleground for AMC expansion via digital channels.
- **Investment insight:** B30 investors often have lower ticket sizes but higher loyalty to SIP, making them ideal for long-tenure wealth products.
- **Trend observation:** The narrowing T30/B30 gap over time reflects India's digital payments revolution enabling mutual fund access in smaller towns.


### Chart 10 — Industry Folio Growth (Line Chart)


In [ ]:
# Chart 10: Industry Folio Growth
folio_sorted = industry_folio.sort_values("month")

fig_folio, ax_folio = plt.subplots(figsize=(14, 6))
ax_folio.plot(
    folio_sorted["month"],
    folio_sorted["total_folios_crore"],
    marker="o",
    linewidth=2.5,
    color="#8e44ad",
    markersize=6,
    label="Total Folios",
)

milestones = {
    "2022-01-01": (13.26, "Jan 2022\n13.26 Cr"),
    "2025-12-01": (26.12, "Dec 2025\n26.12 Cr"),
}
for date_str, (val, label) in milestones.items():
    dt = pd.Timestamp(date_str)
    ax_folio.annotate(
        label,
        xy=(dt, val),
        xytext=(dt, val + 1.8),
        fontsize=10,
        fontweight="bold",
        ha="center",
        arrowprops=dict(arrowstyle="->", color="#8e44ad"),
        bbox=dict(boxstyle="round,pad=0.3", facecolor="#f5eef8", edgecolor="#8e44ad"),
    )

ax_folio.set_title("Industry Folio Count Growth (2022–2025)", pad=15)
ax_folio.set_xlabel("Month")
ax_folio.set_ylabel("Total Folios (Crore)")
ax_folio.legend()
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
save_matplotlib_figure(fig_folio, "10_industry_folio_growth.png")
plt.show()


#### Insight

- **Business insight:** Industry folios nearly doubled from 13.26 crore (Jan 2022) to 26.12 crore (Dec 2025), validating the industry's retail expansion strategy.
- **Investment insight:** A growing folio base creates a sticky recurring revenue stream for AMCs through trail commissions and cross-selling opportunities.
- **Trend observation:** The growth curve is convex — acceleration post-2023 aligns with the SIP boom and simplified KYC via Aadhaar-based onboarding.


### Chart 11 — Daily Return Correlation (10 Schemes)


In [ ]:
# Chart 11: Daily Return Correlation Heatmap
selected_codes = fund_master.drop_duplicates("category").head(10)["amfi_code"].tolist()
if len(selected_codes) < 10:
    extra = fund_master[~fund_master["amfi_code"].isin(selected_codes)]["amfi_code"].head(10 - len(selected_codes))
    selected_codes.extend(extra.tolist())

nav_selected = nav_history[nav_history["amfi_code"].isin(selected_codes)].copy()
nav_pivot = nav_selected.pivot_table(index="date", columns="amfi_code", values="nav")
daily_returns = nav_pivot.pct_change().dropna()

code_to_name = fund_master.set_index("amfi_code")["scheme_name"].to_dict()
short_names = {c: code_to_name.get(c, str(c))[:30] for c in daily_returns.columns}
daily_returns.columns = [short_names[c] for c in daily_returns.columns]

corr_matrix = daily_returns.corr()

fig_corr, ax_corr = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    vmin=-1,
    vmax=1,
    square=True,
    linewidths=0.5,
    ax=ax_corr,
    cbar_kws={"label": "Correlation"},
    annot_kws={"size": 9},
)
ax_corr.set_title("Daily Return Correlation — 10 Selected Schemes", pad=15)
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
save_matplotlib_figure(fig_corr, "11_daily_return_correlation.png")
plt.show()


#### Insight

- **Business insight:** High correlations (>0.85) among large-cap and flexi-cap funds mean product differentiation is limited — AMCs must compete on brand, service, and expense ratio.
- **Investment insight:** Adding low-correlation assets (debt, gold, international) to a portfolio of Indian equity MFs can meaningfully improve diversification benefits.
- **Trend observation:** Small-cap and mid-cap schemes show slightly lower correlation with large-cap peers, offering genuine diversification within the equity sleeve.


### Chart 12 — Sector Allocation (Donut Chart)


In [ ]:
# Chart 12: Sector Allocation Donut Chart
sector_weights = (
    portfolio_holdings.groupby("sector")["weight_pct"]
    .sum()
    .sort_values(ascending=False)
)
sector_weights = (sector_weights / sector_weights.sum() * 100).round(2)

fig_sector, ax_sector = plt.subplots(figsize=(10, 10))
colors_sector = sns.color_palette("Spectral", len(sector_weights))
_, _, autotexts = ax_sector.pie(
    sector_weights,
    labels=sector_weights.index,
    autopct="%1.1f%%",
    startangle=90,
    colors=colors_sector,
    pctdistance=0.78,
    wedgeprops=dict(width=0.45, edgecolor="white", linewidth=1.5),
)
ax_sector.set_title("Aggregate Sector Allocation Across Portfolio Holdings", pad=15)
for t in autotexts:
    t.set_fontsize(8)
plt.tight_layout()
save_matplotlib_figure(fig_sector, "12_sector_allocation_donut.png")
plt.show()


#### Insight

- **Business insight:** Banking and IT together account for the largest sector weights, reflecting index composition bias — AMCs with active sector bets must justify deviation from benchmark.
- **Investment insight:** Over-concentration in Financials and IT creates sector-specific risk; investors should monitor quarterly portfolio disclosures for drift.
- **Trend observation:** Pharma and FMCG provide defensive ballast in portfolios, balancing cyclical exposure from Banking and Auto sectors.


### Chart 13 — Top 10 Performing Funds (Bar Chart)


In [ ]:
# Chart 13: Top 10 Performing Funds by 3-Year Return
top10 = (
    scheme_performance.nlargest(10, "return_3yr_pct")
    .sort_values("return_3yr_pct", ascending=True)
    .copy()
)
top10["short_name"] = (
    top10["scheme_name"]
    .str.replace(" - Regular Plan - Growth", "", regex=False)
    .str.replace(" - Direct Plan - Growth", " (D)", regex=False)
    .str.replace(" - Regular - Growth", "", regex=False)
    .str[:35]
)

fig_top, ax_top = plt.subplots(figsize=(12, 7))
bars = ax_top.barh(
    top10["short_name"],
    top10["return_3yr_pct"],
    color=sns.color_palette("rocket", len(top10)),
    edgecolor="white",
)
ax_top.set_title("Top 10 Mutual Funds by 3-Year Return (%)", pad=15)
ax_top.set_xlabel("3-Year Return (%)")
ax_top.set_ylabel("")
for bar, val in zip(bars, top10["return_3yr_pct"]):
    ax_top.text(
        bar.get_width() + 0.3,
        bar.get_y() + bar.get_height() / 2,
        f"{val:.1f}%",
        va="center",
        fontsize=10,
        fontweight="bold",
    )
plt.tight_layout()
save_matplotlib_figure(fig_top, "13_top10_performing_funds.png")
plt.show()


#### Insight

- **Business insight:** Small-cap funds dominate the top-10 list with 3-year returns exceeding 20%, attracting aggressive retail flows but also raising SEBI's concern on frothy valuations.
- **Investment insight:** Outperformance in small-cap comes with higher volatility and drawdowns — investors must match fund risk profiles to their time horizon and tolerance.
- **Trend observation:** SBI and Nippon India small-cap funds consistently rank among top performers, reflecting strong stock-selection capabilities in the mid/small-cap universe.


### Chart 14 — Risk vs Return (Scatter Plot)


In [ ]:
# Chart 14: Risk vs Return Scatter (Bubble size = AUM)
perf_plot = scheme_performance.dropna(subset=["std_dev_ann_pct", "return_3yr_pct", "aum_crore"])

fig_risk, ax_risk = plt.subplots(figsize=(12, 8))
scatter = ax_risk.scatter(
    perf_plot["std_dev_ann_pct"],
    perf_plot["return_3yr_pct"],
    s=perf_plot["aum_crore"] / 30,
    c=perf_plot["return_3yr_pct"],
    cmap="RdYlGn",
    alpha=0.75,
    edgecolors="grey",
    linewidth=0.5,
)

for _, row in perf_plot.nlargest(5, "return_3yr_pct").iterrows():
    short = row["scheme_name"][:25]
    ax_risk.annotate(
        short,
        (row["std_dev_ann_pct"], row["return_3yr_pct"]),
        fontsize=7,
        alpha=0.85,
        xytext=(5, 5),
        textcoords="offset points",
    )

ax_risk.set_title("Risk vs Return — Mutual Fund Schemes (Bubble Size = AUM)", pad=15)
ax_risk.set_xlabel("Annualised Standard Deviation (%)")
ax_risk.set_ylabel("3-Year Return (%)")
plt.colorbar(scatter, ax=ax_risk, label="3Y Return (%)")
plt.tight_layout()
save_matplotlib_figure(fig_risk, "14_risk_vs_return_scatter.png")
plt.show()


#### Insight

- **Business insight:** The risk-return frontier shows a clear positive slope — higher-volatility small/mid-cap funds deliver superior returns, validating the risk premium in Indian equities.
- **Investment insight:** Large-AUM large-cap funds cluster in the lower-left (low risk, moderate return), ideal for core portfolio allocation with satellite small-cap exposure.
- **Trend observation:** Schemes with Sharpe ratios above 1.0 offer the best risk-adjusted choices for quality-focused investors.


### Chart 15 — Benchmark vs Average Mutual Fund NAV


In [ ]:
# Chart 15: Benchmark Performance vs Average MF NAV
nav_daily_avg = nav_history.groupby("date")["nav"].mean().reset_index()
base_nav = nav_daily_avg["nav"].iloc[0]
nav_daily_avg["normalised_nav"] = nav_daily_avg["nav"] / base_nav * 100

nifty = benchmark_indices[benchmark_indices["index_name"] == "NIFTY50"].copy().sort_values("date")
base_nifty = nifty["close_value"].iloc[0]
nifty["normalised_index"] = nifty["close_value"] / base_nifty * 100

fig_bench = go.Figure()
fig_bench.add_trace(go.Scatter(
    x=nav_daily_avg["date"],
    y=nav_daily_avg["normalised_nav"],
    mode="lines",
    name="Avg MF NAV (Base 100)",
    line=dict(color="#2980b9", width=2),
))
fig_bench.add_trace(go.Scatter(
    x=nifty["date"],
    y=nifty["normalised_index"],
    mode="lines",
    name="NIFTY 50 (Base 100)",
    line=dict(color="#e74c3c", width=2, dash="dot"),
))

fig_bench.update_layout(
    title="Benchmark vs Average Mutual Fund NAV Performance (Normalised to 100)",
    xaxis_title="Date",
    yaxis_title="Normalised Value (Base = 100)",
    template=PLOTLY_TEMPLATE,
    height=550,
    legend=dict(x=0.02, y=0.98),
    hovermode="x unified",
)
fig_bench.show()
save_plotly_figure(fig_bench, "15_benchmark_vs_avg_nav.html")


#### Insight

- **Business insight:** The average MF NAV closely tracks NIFTY 50, confirming that the industry's equity-heavy AUM base moves in tandem with broad market indices.
- **Investment insight:** Periods where average NAV outperforms NIFTY suggest active management alpha from mid/small-cap overweight — a case for diversified flexi-cap funds over pure index replication.
- **Trend observation:** Both series show synchronized dips during the 2024 correction, reinforcing the importance of asset allocation and rebalancing in volatile cycles.


---

## 6. Key Takeaways & Next Steps

| Theme | Key Finding |
|-------|-------------|
| **Industry Growth** | AUM and folio counts doubled; SIP inflows hit 31,002 Cr/month by Dec 2025 |
| **Market Leadership** | SBI MF leads with 12.5L Cr AUM; top-3 AMCs control ~40% of tracked assets |
| **Investor Profile** | Young (26–35), male-dominated, T30-heavy but B30 share rising |
| **Performance** | Small-cap funds lead 3Y returns; risk-return trade-off is well-defined |
| **Portfolio** | Banking + IT dominate sector allocation; diversification within equity is limited |
| **Benchmark** | Active MF NAV closely tracks NIFTY 50 with occasional alpha pockets |

### Recommended Next Steps
1. **Time-series modelling** — Forecast SIP inflows and AUM using ARIMA/Prophet
2. **Cluster analysis** — Segment schemes by risk-return profiles for portfolio construction
3. **SQL analytics layer** — Load cleaned data into SQLite/PostgreSQL for ad-hoc querying (see `sql/` folder)
4. **Interactive dashboard** — Build a Streamlit/Dash app for live KPI monitoring
5. **Statistical testing** — Test whether small-cap outperformance is statistically significant vs large-cap

---

*All 15 charts exported to `../reports/charts/` — Plotly figures as `.html`, Matplotlib/Seaborn figures as `.png`.*


In [ ]:
# =============================================================================
# Export Summary — List all saved chart files
# =============================================================================
saved_charts = sorted(CHARTS_DIR.glob("*"))
print(f"Total chart files in {CHARTS_DIR}:\n")
for f in saved_charts:
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name:<45} ({size_kb:,.1f} KB)")
